# GPU benchmark for contextual data classification

Use **Runtime → Change runtime type → GPU**, then **Run all**. This notebook requires an NVIDIA GPU with at least 12 GiB visible VRAM (a 16 GB T4 is the intended starting point) and 22 GiB free disk. Set a Colab secret named `GITHUB_TOKEN` with read access to the private Data Discovery repository and enable notebook access. No Hugging Face token, paid inference API, or external model endpoint is required.

The notebook clones the pinned pushed revision and applies the included, SHA-256-checked snapshot of the later reliability engine and evaluation helpers. The snapshot is bundled because these changes have not been pushed. It runs Qwen2.5-7B-Instruct on the Colab GPU with 4-bit NF4. The existing Presidio, Saudi validators and MiniLM run locally on the Colab CPU. Synthetic benchmark labels stay separate from inference. Only allowlisted metadata, aggregate structural evidence, and declared relationships reach the model.

The diagnostic set is the original demo and all nine previously evaluated sectors. The final comparison uses 72 fresh fields across agriculture, construction, and museum services. One real contextual proposal per table supplies the LLM-only result and the hybrid's second opinion. The hybrid reuses the existing engine and keeps conservative validation and review. The production pipeline is unchanged.

[Model](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct) · [NF4 integration](https://huggingface.co/docs/transformers/v4.49.0/quantization/bitsandbytes) · [Constrained JSON generation](https://pypi.org/project/lm-format-enforcer/0.11.3/)


In [ ]:
import base64
import hashlib
import io
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import zipfile

from google.colab import userdata

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Select a GPU runtime in Colab before running.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                "--format=csv,noheader"], check=True)
if shutil.disk_usage("/content").free < 22 * 1024**3:
    raise RuntimeError("This benchmark needs at least 22 GiB of free disk for weights and dependencies.")

AUTO_DOWNLOAD = True
REPO = "https://github.com/ghaida-alsalamah/data-discovery-classification.git"
BASE = "dd58a8cbc040792aaa5d0d7eeb4c76d0b8e2c71c"
ROOT = Path(tempfile.mkdtemp(prefix="data-discovery-", dir="/content"))
REPO_DIR = ROOT / "repository"
try:
    secret = userdata.get("GITHUB_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    raise RuntimeError("Add the GITHUB_TOKEN Colab secret and allow notebook access.") from None
if not secret:
    raise RuntimeError("GITHUB_TOKEN is empty.")

# Do not place the secret in the URL, git config, subprocess arguments, or notebook output.
askpass = ROOT / "askpass"
askpass.write_text('''#!/bin/sh
case "$1" in
  *Username*) printf '%s\n' x-access-token ;;
  *) printf '%s\n' "$COLAB_GIT_TOKEN" ;;
esac
''')
askpass.chmod(0o700)
auth = dict(os.environ, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0",
            COLAB_GIT_TOKEN=secret)
try:
    commands = [
        ["git", "-c", "credential.helper=", "clone", "--quiet", "--no-checkout", REPO, str(REPO_DIR)],
        ["git", "-C", str(REPO_DIR), "checkout", "--quiet", "--detach", BASE],
    ]
    for command in commands:
        if subprocess.run(command, env=auth, stdout=subprocess.DEVNULL,
                          stderr=subprocess.DEVNULL).returncode:
            raise RuntimeError("Could not clone or check out the pinned private repository. Check the secret's read access.")
finally:
    askpass.unlink(missing_ok=True)
    del auth, secret

# Snapshot of the local post-Phase-3 files for the experiment.
OVERLAY = "..."
OVERLAY_SHA = "b197d3a13e2c7e587a48598ba32e0123edc4f57ca8d4d5a5e45149172bf5bda0"
overlay = base64.b64decode(OVERLAY, validate=True)
if hashlib.sha256(overlay).hexdigest() != OVERLAY_SHA:
    raise ValueError("The bundled evaluation snapshot failed its integrity check.")
with zipfile.ZipFile(io.BytesIO(overlay)) as archive:
    for info in archive.infolist():
        target = (REPO_DIR / info.filename).resolve()
        if not target.is_relative_to(REPO_DIR) or target.suffix not in {".py", ".md"}:
            raise ValueError("Invalid bundled path.")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(info))
del overlay, OVERLAY
print(f"Checked out {BASE[:8]} and applied the bundled snapshot.")


In [ ]:
# Inherit Colab's CUDA-enabled PyTorch. Install benchmark packages in an isolated environment.
venv = ROOT / "venv"
subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(venv)], check=True)
PYTHON = str(venv / "bin/python")
subprocess.run([PYTHON, "-m", "pip", "install", "-q", "-r", "backend/requirements.txt",
                "transformers==4.57.6", "accelerate==1.11.0", "bitsandbytes==0.48.2",
                "huggingface-hub==0.36.2", "lm-format-enforcer==0.11.3", "spacy==3.8.18",
                "numpy==2.2.6", "tokenizers==0.22.2"], cwd=REPO_DIR, check=True)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MODEL_REV = "a09a35458c702b33eeacc393d103063234e8bc28"
MINILM_REV = "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
subprocess.run([PYTHON, "-c", """
import torch
assert torch.cuda.is_available(), 'The venv needs Colab CUDA-enabled PyTorch.'
props = torch.cuda.get_device_properties(0)
assert props.total_memory >= 12 * 1024**3, 'Select a GPU with at least 12 GiB VRAM.'
assert torch.cuda.get_device_capability(0)[0] >= 7, 'Select a T4 or newer NVIDIA GPU.'
print(props.name, round(props.total_memory / 1024**3, 1), 'GiB of GPU memory')
"""], cwd=REPO_DIR, check=True)

env = dict(os.environ, HF_HUB_DISABLE_TELEMETRY="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1",
           TOKENIZERS_PARALLELISM="false", COLAB_OVERLAY_SHA=OVERLAY_SHA, COLAB_BASE=BASE)
subprocess.run([PYTHON, "-c", """
import sys
from huggingface_hub import snapshot_download
snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], local_dir='.models/qwen-7b',
                  allow_patterns=['*.safetensors', '*.json', 'merges.txt', 'vocab.json', 'LICENSE'],
                  token=False, max_workers=4)
snapshot_download(repo_id='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
                  revision=sys.argv[3], local_dir='.models/minilm',
                  allow_patterns=['tokenizer.json', 'onnx/model.onnx'], token=False)
""", MODEL_ID, MODEL_REV, MINILM_REV], cwd=REPO_DIR, env=env, check=True)
env.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
print("Pinned weights downloaded. The inference step uses local files.")


## Run the experiment

The subprocess performs a real constrained GPU smoke call, then processes the diagnostic databases. It seals the engine, prompt, scorer, and model revision before generating and scoring the fresh set. Invalid model responses become abstentions for the LLM-only arm and use the existing classifier for the hybrid. Counts are exported. Model text, prompts, and input SQLite files are excluded from the download.

In [ ]:
RUN_TESTS = True
runner_path = REPO_DIR / "evaluation/colab_trial.py"
runner_path.write_text(r'''import csv
import hashlib
import json
import os
import resource
import sqlite3
import sys
from collections import Counter
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import torch
from lmformatenforcer import JsonSchemaParser
from lmformatenforcer.integrations.transformers import (
    build_token_enforcer_tokenizer_data, build_transformers_prefix_allowed_tokens_fn,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from backend.app.classification.ai_evidence import PresidioEvidence, SemanticEvidence
from backend.app.classification.engine import HybridClassifier
from backend.app.classification.local_llm import (
    DIRECT, DOMAINS_ALLOWED, GUIDE, SEMANTICS, classify_table, response_schema, validate_response,
)
from backend.app.connectors.sqlite import InformationAsset, SQLiteConnector
from backend.app.discovery.schema import discover
from backend.app.intelligence.service import analyze
from backend.app.models.synthetic import generate_database
from backend.app.profiling.columns import profile_database
from evaluation import final_holdout, fixtures, local_llm_holdout, metacrafter_holdout
from evaluation.reliability_infer import fingerprint
from evaluation.reliability_score import measured
from evaluation.score import phase3, rows_for

MODEL_REV = "a09a35458c702b33eeacc393d103063234e8bc28"
MINILM_REV = "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"


def digest(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def save(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, allow_nan=False, indent=2) + "\n", encoding="utf-8")


def load(path):
    return json.loads(path.read_text(encoding="utf-8"))


class GPU:
    def __init__(self):
        if not torch.cuda.is_available():
            raise RuntimeError("This benchmark requires a CUDA GPU.")
        torch.manual_seed(42)
        torch.cuda.reset_peak_memory_stats()
        started = perf_counter()
        self.dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        model_path = ".models/qwen-7b"
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True,
                                                       trust_remote_code=False)
        self.constraints = build_token_enforcer_tokenizer_data(self.tokenizer)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path, local_files_only=True, trust_remote_code=False, use_safetensors=True,
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=self.dtype),
            dtype=self.dtype, device_map={"": 0}, attn_implementation="sdpa",
        ).eval()
        if not self.model.is_loaded_in_4bit or self.model.get_input_embeddings().weight.device.type != "cuda":
            raise RuntimeError("The benchmark did not load the intended 4-bit CUDA model.")
        torch.cuda.synchronize()
        self.load_seconds = perf_counter() - started
        self.totals = Counter()

    def classify(self, payload):
        fields = payload["fields"]
        prompt = json.dumps({"types": SEMANTICS, "domains": DOMAINS_ALLOWED, **payload},
                            ensure_ascii=False, separators=(",", ":"))
        messages = [{"role": "system", "content": GUIDE}, {"role": "user", "content": prompt}]
        inputs = self.tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                    return_tensors="pt", return_dict=True).to("cuda:0")
        size = inputs["input_ids"].shape[-1]
        max_new = 36 * len(fields) + 64
        if size + max_new > 4096:
            raise ValueError("Prompt exceeds the experiment context cap.")
        grammar = build_transformers_prefix_allowed_tokens_fn(
            self.constraints, JsonSchemaParser(response_schema(fields)))
        torch.cuda.synchronize()
        started = perf_counter()
        with torch.inference_mode():
            output = self.model.generate(**inputs, max_new_tokens=max_new, do_sample=False,
                prefix_allowed_tokens_fn=grammar, pad_token_id=self.tokenizer.eos_token_id)
        torch.cuda.synchronize()
        answer = output[0, size:]
        self.totals.update(requests=1, generation_seconds=perf_counter() - started,
                           prompt_tokens=size, generated_tokens=len(answer))
        # The inference response is validated in memory and never written or printed.
        return validate_response(json.loads(self.tokenizer.decode(answer, skip_special_tokens=True)), fields)


def prepare(work, scope):
    root = work / scope
    db_root, label_root = root / "databases", root / "labels"
    db_root.mkdir(parents=True)
    label_root.mkdir()
    if scope == "diagnostics":
        generate_database(db_root / "synthetic.sqlite", label_root / "synthetic.json")
        general = work / "generalization-source"
        fixtures.generate(general)
        for name in ("healthcare", "ecommerce", "education"):
            (db_root / f"{name}.sqlite").write_bytes((general / "databases" / f"{name}.sqlite").read_bytes())
            (label_root / f"{name}.json").write_bytes((general / "labels" / f"{name}.json").read_bytes())
        for name in final_holdout.DATA:
            save(label_root / f"{name}.json", final_holdout.build(db_root / f"{name}.sqlite", name))
        for name in metacrafter_holdout.CATALOG:
            save(label_root / f"{name}.json", metacrafter_holdout.build(name, db_root / f"{name}.sqlite"))
    else:
        for name, (seed, spec) in local_llm_holdout.SECTORS.items():
            save(label_root / f"{name}.json", local_llm_holdout.build(db_root / f"{name}.sqlite", seed, spec))
    files = sorted([*db_root.glob("*.sqlite"), *label_root.glob("*.json")])
    manifest = {p.relative_to(root).as_posix(): digest(p) for p in files}
    save(root / "manifest.json", manifest)
    return root


def infer(databases, output, model, presidio, gpu):
    for index, db_path in enumerate(databases, 1):
        with SQLiteConnector(db_path, InformationAsset(db_path.stem, "Synthetic benchmark", "Evaluation")).connect() as conn:
            started = perf_counter()
            schema, links = discover(conn)
            profiles = profile_database(conn, schema, 500, presidio)
            profiled = perf_counter()
            baseline = model.classify(schema, profiles)
            classified = perf_counter()
            index_rows = {(row["table_name"], row["column_name"]): row for row in baseline}
            local, hybrid, statuses = [], [], Counter()
            for table in schema["tables"]:
                name = table["name"]
                current = {col["name"]: index_rows[name, col["name"]] for col in table["columns"]}
                standalone, combined, status = classify_table(gpu, schema, table, profiles[name], current)
                local.extend(standalone)
                hybrid.extend(combined)
                statuses[status] += 1
            reasoned = perf_counter()
            variants = {}
            for name, rows in (("current", baseline), ("local_llm", local), ("hybrid", hybrid)):
                domains, quality, opportunities = analyze(conn, schema, links, profiles, rows, use_environment=False)
                variants[name] = {"hybrid": rows, "schema": schema, "domains": domains,
                                  "quality": quality, "opportunities": opportunities}
        save(output / f"{db_path.stem}.json", {"variants": variants, "requests": dict(statuses),
             "seconds": {"profiling": profiled - started, "current_classification": classified - profiled,
                         "shared_llm_and_hybrid": reasoned - classified}})
        print(f"Database {index}/{len(databases)}: {dict(statuses)}", flush=True)


def score(root, output, scope):
    manifest = load(root / "manifest.json")
    if any(digest(root / name) != expected for name, expected in manifest.items()):
        raise ValueError("Benchmark input or label hash changed.")
    variants = {name: {"sectors": {}, "phase3": {}, "_rows": []}
                for name in ("current", "local_llm", "hybrid")}
    flat, failures, times = [], Counter(), Counter()
    for path in sorted((root / "labels").glob("*.json")):
        report = load(output / "predictions" / path.name)
        labels = load(path)
        save(output / "labels" / path.name, labels)
        failures.update(report["requests"])
        times.update(report["seconds"])
        for name, result in report["variants"].items():
            rows = rows_for(path.stem, labels, result["hybrid"])
            variants[name]["sectors"][path.stem] = measured(rows)
            variants[name]["phase3"][path.stem] = phase3(result)
            variants[name]["_rows"].extend(rows)
            flat.extend({**row, "engine": name} for row in rows)
        audit(root / "databases" / f"{path.stem}.sqlite", labels,
              (output / "predictions" / path.name).read_text(encoding="utf-8"))
    if not failures["completed"]:
        raise RuntimeError("No valid real model response. Do not interpret these as LLM results.")
    for result in variants.values():
        result["combined"] = measured(result.pop("_rows"))
    counts = {item["combined"]["fields"] for item in variants.values()}
    if counts != ({472} if scope == "diagnostics" else {72}):
        raise ValueError(f"Unexpected benchmark field counts: {counts}")
    with (output / "classifications.csv").open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(flat[0]))
        writer.writeheader()
        writer.writerows(flat)
    save(output / "metrics.json", {"scope": scope, "request_status": dict(failures),
                                    "seconds": dict(times), "variants": variants})
    save(output / "fixture_manifest.json", manifest)


def audit(db, labels, exported):
    def quote(name):
        return '"' + name.replace('"', '""') + '"'
    with sqlite3.connect(db.resolve().as_uri() + "?mode=ro", uri=True) as conn:
        for table, columns in labels.items():
            for column, row in columns.items():
                if row["semantic_type"] in DIRECT:
                    query = f"SELECT DISTINCT {quote(column)} FROM {quote(table)}"
                    if any(isinstance(value, str) and len(value) >= 10 and value in exported
                           for (value,) in conn.execute(query)):
                        raise ValueError("Raw synthetic direct identifiers found in export.")


def seal():
    tracked = [
        Path(__file__), Path("evaluation/DOMAIN_SCORING.md"), Path("evaluation/score.py"),
        Path("evaluation/reliability_score.py"), Path("evaluation/local_llm_holdout.py"),
        Path("backend/requirements.txt"),
    ]
    return {
        "backend": fingerprint(Path(".")),
        "experiment_files": {path.as_posix(): digest(path) for path in tracked},
        "base_commit": os.environ["COLAB_BASE"], "overlay_sha256": os.environ["COLAB_OVERLAY_SHA"],
        "model": "Qwen/Qwen2.5-7B-Instruct", "model_revision": MODEL_REV, "minilm_revision": MINILM_REV,
        "quantization": "bitsandbytes NF4 with double quantization", "sample_cap": 500,
        "seed": 42, "context_tokens": 4096,
        "packages": {name: version(name) for name in ("torch", "transformers", "accelerate", "bitsandbytes",
                     "lm-format-enforcer", "onnxruntime", "presidio-analyzer", "spacy")},
    }


def main(work, export):
    fixed = seal()
    pres = PresidioEvidence()
    model = HybridClassifier(SemanticEvidence(Path(".models/minilm")))
    gpu = GPU()
    # A real constrained GPU call fails fast on incompatible CUDA/model/schema setups.
    gpu.classify({"table": "t0", "hint": "customer", "fields": {"f0": {
        "hint": "email", "store": "text", "det": {"email_address": [10, 0]},
        "pk": False, "fk": []}}})
    gpu.totals.clear()
    for scope in ("diagnostics", "fresh"):
        if seal() != fixed:
            raise RuntimeError("Classifier or evaluation configuration changed.")
        if scope == "fresh":
            save(export / "holdout_seal.json", fixed)
        print(f"Running {scope}", flush=True)
        root = prepare(work, scope)
        output = export / scope
        before = gpu.totals.copy()
        infer(sorted((root / "databases").glob("*.sqlite")), output / "predictions", model, pres, gpu)
        if seal() != fixed:
            raise RuntimeError("Configuration changed during inference.")
        score(root, output, scope)
        usage = {key: gpu.totals[key] - before[key] for key in gpu.totals}
        save(output / "resources.json", {**usage,
             "generated_tokens_per_second": usage.get("generated_tokens", 0) /
                 max(usage.get("generation_seconds", 0), .001),
             "gpu_peak_allocated_mib_cumulative": torch.cuda.max_memory_allocated() / 2**20,
             "gpu_peak_reserved_mib_cumulative": torch.cuda.max_memory_reserved() / 2**20,
             "host_peak_rss_mib_cumulative": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024})
    props = torch.cuda.get_device_properties(0)
    save(export / "provenance.json", {**fixed, "gpu": props.name, "gpu_total_mib": props.total_memory / 2**20,
         "cuda": torch.version.cuda, "compute_dtype": str(gpu.dtype), "model_load_seconds": gpu.load_seconds})


if __name__ == "__main__":
    main(Path(sys.argv[1]), Path(sys.argv[2]))
''', encoding="utf-8")
if RUN_TESTS:
    subprocess.run([PYTHON, "-m", "pytest", "-q", "tests"], cwd=REPO_DIR, env=env, check=True)

EXPORT = ROOT / "results"
subprocess.run([PYTHON, "-m", "evaluation.colab_trial", str(ROOT / "inputs"), str(EXPORT)],
               cwd=REPO_DIR, env=env, check=True)


In [ ]:
import json
import pandas as pd
from IPython.display import display
from google.colab import files

for scope in ("diagnostics", "fresh"):
    record = json.loads((EXPORT / scope / "metrics.json").read_text())
    rows = []
    for name, variant in record["variants"].items():
        metric = variant["combined"]
        rows.append({
            "engine": name,
            "fields": metric["fields"],
            "semantic exact": metric["strict_exact_accuracy"],
            "semantic category": metric["common_category_accuracy"],
            "domain exact": metric["strict_domain_accuracy"],
            "domain hierarchy": metric["hierarchical_domain_accuracy"],
            "personal precision": metric["personal"]["precision"],
            "personal recall": metric["personal"]["recall"],
            "personal F1": metric["personal"]["f1"],
            "personal FN": metric["personal"]["fn"],
            "unknown F1": metric["unknown"]["f1"],
            "review rate": metric["review_rate"],
            "automated coverage": metric["automated_coverage"],
            "unreviewed exact": metric["unreviewed_strict_semantic_accuracy"],
            "high-score errors": metric["high_confidence_strict_error"]["count"],
            "high-score predictions": metric["high_confidence_strict_error"]["denominator"],
        })
    print(scope, record["request_status"], record["seconds"])
    display(pd.DataFrame(rows).round(3))
    print(json.loads((EXPORT / scope / "resources.json").read_text()))

archive = shutil.make_archive(str(ROOT / "data-discovery-colab-results"), "zip", root_dir=EXPORT)
print("Results archive:", archive)
if AUTO_DOWNLOAD:
    files.download(archive)
